In [0]:
#%run ./transform_data ----- A décommmenter pour lancer les notebooks séparements

In [0]:
fact_batch_note = batch_note.select(
    "batch",
    "id_batch_note",
    "note",
    "gap_minutes",
    "location",
    "impact",
    "event",
    "detail",
    "event_date",
    "profile_user",
    F.col("created_at").alias("date_saisie")
).filter(F.col("deleted") == False)

In [0]:
fact_batch_note = fact_batch_note.withColumn(
    "entry_date",
    to_date("date_saisie")
)

In [0]:
fact_batch_note_with_id_plant = fact_batch_note.alias("a").join(
    batches_info.alias("b"),
    F.col("a.batch") == F.col("b.batch_id"),
    "left"
).select(
        "a.*",
        "b.id_plant_production_line"
    )

In [0]:
fact_batch_note_with_profile_user = (fact_batch_note_with_id_plant.alias("a").join(
    profiles_users.alias("b"),
    F.col("a.profile_user") == F.col("b.id_profile_user")
).select(
        "a.*",
        F.concat_ws(" ", F.col("b.name"), F.col("b.surname")).alias("full_name")
    )
)

In [0]:
fact_batch_note_with_profile_user = fact_batch_note_with_profile_user.select(
    "batch",
    "id_batch_note",
    "note",
    "gap_minutes",
    "location",
    "impact",
    "event",
    "detail",
    "event_date",
    "date_saisie",
    "entry_date",
    "id_plant_production_line",
    "full_name"
)

In [0]:
current_process= "fact_batch_note"

In [0]:
target_fact_batch_note = current_catalog +"."+current_schema+"."+current_process
print(target_fact_batch_note)

In [0]:
all_columns =  fact_batch_note_with_profile_user.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = ['id_batch_note']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    fact_batch_note_with_profile_user, 
    target_fact_batch_note, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode  # Use "update" for update mode, "full" for delete/insert mode
    )